# Парсинг сайтов 

In [2]:
from bs4 import BeautifulSoup as bs

In [3]:
import pandas as pd

In [4]:
from curl_cffi import requests

In [5]:
import time

In [6]:
import random

In [7]:
import re

In [8]:
import os

# Получение информации

In [9]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Linux; Android 13; SM-G991B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.6099.144 Mobile Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Sec-Fetch-Site': 'same-origin',
}

In [10]:
url = 'https://om.kinogo-filmov.net/top250/'

In [11]:
page = requests.get(url, headers=headers, verify=False, timeout=30, impersonate="chrome120")

In [12]:
page.status_code

200

In [13]:
soup = bs(page.text, 'html.parser')

In [14]:
page.text

'<!doctype html>\n<html lang="ru-RU">\n<head>\n<title>250 лучших фильмов » КиноГо 24 - смотреть онлайн фильмы, мультфильмы, сериалы на Андроид, Айфоне и Айпаде на киного смотреть онлайн бесплатно в хорошем качестве hd 720</title>\n<meta charset="utf-8">\n<meta name="description" content="Рейтинг формируется на основе голосов пользователей кинотеатра онлайн. Каждый посетитель может выразить своё мнение, отдав голос за любимый фильм.">\n<meta name="keywords" content="250 лучших фильмов, онлайн кинотеатр, рейтинг фильмов, голосование пользователей, популярные фильмы, любимые фильмы, кино онлайн, топ фильмов, пользовательские рейтинги, лучшие картины, фильмы по мнению зрителей, кинообзоры, кино рейтинги, выбор зрителей, фильмы для просмотра.">\n<meta name="generator" content="DataLife Engine (https://dle-news.ru)">\n<link rel="canonical" href="https://om.kinogo-filmov.net/top250/">\n<link rel="alternate" type="application/rss+xml" title="250 лучших фильмов » КиноГо 24 - смотреть онлайн фил

In [15]:
soup

<!DOCTYPE html>

<html lang="ru-RU">
<head>
<title>250 лучших фильмов » КиноГо 24 - смотреть онлайн фильмы, мультфильмы, сериалы на Андроид, Айфоне и Айпаде на киного смотреть онлайн бесплатно в хорошем качестве hd 720</title>
<meta charset="utf-8"/>
<meta content="Рейтинг формируется на основе голосов пользователей кинотеатра онлайн. Каждый посетитель может выразить своё мнение, отдав голос за любимый фильм." name="description"/>
<meta content="250 лучших фильмов, онлайн кинотеатр, рейтинг фильмов, голосование пользователей, популярные фильмы, любимые фильмы, кино онлайн, топ фильмов, пользовательские рейтинги, лучшие картины, фильмы по мнению зрителей, кинообзоры, кино рейтинги, выбор зрителей, фильмы для просмотра." name="keywords"/>
<meta content="DataLife Engine (https://dle-news.ru)" name="generator"/>
<link href="https://om.kinogo-filmov.net/top250/" rel="canonical"/>
<link href="https://om.kinogo-filmov.net/top250/rss.xml" rel="alternate" title="250 лучших фильмов » КиноГо 24 -

In [16]:
result_list = {'title': [], 'description': [], 'date': [], 'views': []}

# Алгоритм

In [17]:
def clean_text(text):
    """Очищает текст от лишних пробелов и переносов"""
    if text:
        return re.sub(r'\s+', ' ', text).strip()
    return None

In [18]:
def parse_movie_regex(html_text):
    """Парсит один фильм через регулярные выражения"""
    
    movie = {}
    
    # Название 
    title_match = re.search(r'<h2 class="zagolovki"><a href="([^"]+)">([^<]+)</a></h2>', html_text)
    if title_match:
        title = title_match.group(2)
        title = re.sub(r'\s*\(\d{4}\)\s*$', '', title)
        movie['name'] = title.strip()
    
    # Год выпуска
    year_match = re.search(r'<b>Год выпуска:</b>\s*<a[^>]*>(\d{4})</a>', html_text)
    movie['year'] = year_match.group(1) if year_match else None

    # Страна
    desc_match = re.search(r'<!--TEnd-->\s*([^<]+(?:<[^>]+>[^<]*)?)', html_text)
    if desc_match:
        description = desc_match.group(1)
        # Убираем "...<br><br>" и лишние пробелы
        description = re.sub(r'\s+', ' ', description).strip()
        # Обрезаем до 300 символов
        movie['description'] = description[:300] + '...' if len(description) > 300 else description
    else:
        movie['description'] = None
    
    # Страна (Выпущено)
    country_match = re.search(r'<b>Выпущено:</b>\s*(.+?)(?=<b>|$)', html_text, re.DOTALL)
    if country_match:
        country_block = country_match.group(1)
        countries = re.findall(r'<a[^>]*>([^<]+)</a>', country_block)
        if countries:
            movie['countries'] = ', '.join(countries)
        else:
            countries_text = re.sub(r'<[^>]+>', '', country_block).strip()
            movie['countries'] = countries_text if countries_text else None
    else:
        movie['countries'] = None
    
        
    # Рейтинг IMDB
    imdb_match = re.search(r'<b>Рейтинг IMDB:</b>\s*([\d.]+)', html_text)
    movie['rating'] = imdb_match.group(1) if imdb_match else None

    # Жанр
    genre_match = re.search(r'<b>Жанр:</b>\s*(.+?)(?=<b>|$)', html_text, re.DOTALL)
    if genre_match:
        genre_block = genre_match.group(1)
        genres = re.findall(r'<a[^>]*>([^<]+)</a>', genre_block)
        if genres:
            movie['genres'] = ', '.join(genres)
        else:
            genres_text = re.sub(r'<[^>]+>', '', genre_block).strip()
            movie['genres'] = genres_text if genres_text else None
    else:
        movie['genres'] = None
    
    return movie

In [24]:
numPage = 1 
all_movies = []
# https://om.kinogo-filmov.net/top250/page/
for i in range(18):
    print("Взлом домбаса")
    page_url = f"https://om.kinogo-filmov.net/top250/page/{numPage}"
    time.sleep(random.uniform(15,30))
    iter_page = requests.get(page_url, headers=headers, verify=False, timeout=10)
    
    if iter_page.status_code != 200:
        print("Ебать пизда, накрыло нахуй")
        print(f'Ошибка: {iter_page.status_code}')
        continue
        
    iter_soup = bs(iter_page.text, 'html.parser')
    items = iter_soup.find_all("div", class_="shortstory")
    
    for item in items:
        movie_html = str(item)
        movie_data = parse_movie_regex(movie_html)
        if (movie_data.get('name')):
            print('Ахуенно')
            all_movies.append(movie_data)
            print(f'фильм {movie_data.get('name')}')
            
    print(len(items))
    print("Страница пропаршена, мани мани уже в юмани")
    numPage += 1
print('!!!!!Парс окончен')

Взлом домбаса
Ахуенно
фильм Проект «Конец света»
Ахуенно
фильм Дьявол носит Prada
Ахуенно
фильм Интерстеллар
Ахуенно
фильм Начало
Ахуенно
фильм Побег из Шоушенка
Ахуенно
фильм F1
Ахуенно
фильм Истребитель демонов: Бесконечная крепость
Ахуенно
фильм Апокалипсис
Ахуенно
фильм Матрица
Ахуенно
фильм Властелин колец: Братство кольца
Ахуенно
фильм Семь
Ахуенно
фильм Бойцовский клуб
Ахуенно
фильм Крестный отец
Ахуенно
фильм Отступники
Ахуенно
фильм Паразиты
Ахуенно
фильм Криминальное чтиво
Ахуенно
фильм Унесённые призраками
Ахуенно
фильм Человек-бензопила. Фильм: История Резе
Ахуенно
фильм В погоне за счастьем
Ахуенно
фильм Оппенгеймер
Ахуенно
фильм Темный рыцарь
Ахуенно
фильм Нэчжа побеждает Царя драконов
Ахуенно
фильм Гарри Поттер и философский камень
Ахуенно
фильм Властелин колец: Возвращение короля
Ахуенно
фильм Аватар
Ахуенно
фильм Волк с Уолл-стрит
Ахуенно
фильм Шрэк
Ахуенно
фильм Зеленая миля
Ахуенно
фильм Мстители: Война бесконечности
Ахуенно
фильм Счастливое число Слевина
Ахуенно
фил

In [25]:
df = pd.DataFrame(all_movies)
df
df = df.head(500)
df

,name,year,description,countries,rating,genres
0,Проект «Конец света»,2026,Астронавт Райленд Грейс просыпается на космиче...,США,8.4,"Драма, Фантастика, Комедия"
1,Дьявол носит Prada,2006,Мечтающая стать журналисткой провинциальная де...,"США, Франция",7,"Драма, Комедия"
2,Интерстеллар,2014,"Когда засуха, пыльные бури и вымирание растени...","США, Великобритания, Канада",8.7,"Драма, Фантастика, Приключения"
3,Начало,2010,"Кобб – талантливый вор, лучший из лучших в опа...","США, Великобритания",8.8,"Фантастика, Боевик, Триллер"
4,Побег из Шоушенка,1994,Бухгалтер Энди Дюфрейн обвинён в убийстве собс...,США,9.3,Драма
...,...,...,...,...,...,...
406,Приключения Реми,2018,Удивительное путешествие по Франции маленького...,"Франция, Бельгия",7.1,"Мелодрама, Приключения, Семейный"
407,Мой Хатико,2023,"Чунцин, начало 2000-х годов. Во время поездки ...",Китай,7.2,Драма
408,Смех и горе у Бела моря,1988,Мультипликационный фильм по произведениям Бори...,None,7.7,"Мультфильмы, Драма, Фэнтези"
409,Сплетение судеб,2023,Смита живет в Индии и мечтает дать своей мален...,"Франция, Канада, Италия",7.2,Драма


In [26]:
df.to_csv('films.csv', index=False, mode="a", header=not (os.path.isfile('films.csv')))